# Imports & Configuration

**What this cell does:** Silences Python `warnings` so the output only shows results and query plans.

**Core concept:** Warnings come from the Python client, not from Spark jobs, so hiding them does not change execution. In production code, filter specific warning categories instead of all of them.

In [0]:
import warnings
warnings.filterwarnings("ignore")

**What this cell does:** Makes IPython display the result of every expression in a cell, not only the last one.

**Core concept:** Several cells below call `show()` and `explain()` in the same cell. Both print directly, but this setting also lets bare expressions (such as a partition count) be displayed alongside them.

In [0]:
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

**What this cell does:** Imports Spark SQL types and the functions module as `F` (`F.col`, `F.split`, `F.sum`, `F.countDistinct`, ...).

**Core concept:** On Databricks the `spark` session is created for you when the notebook attaches to a cluster. This cluster uses standard access mode, which runs on **Spark Connect**: your Python code builds a plan and sends it to the Spark driver, which analyzes, optimizes and executes it. `explain()` asks the driver to return those plans as text.

In [0]:
from pyspark.sql.types import *
import pyspark.sql.functions as F
# SparkSession is pre-created on Databricks as `spark`; no need to import/build it

**What this cell does:** Defines `DATA_PATH`, the Unity Catalog volume holding the input parquet files. The original local `SparkSession.builder` / `sparkContext` code is not needed on Databricks.

**Core concept:** Settings such as `master`, driver memory and log level belong to the cluster, not the notebook. Session-level SQL settings (e.g. `spark.sql.autoBroadcastJoinThreshold`, used later) can still be changed with `spark.conf.set`.

In [0]:
# On Databricks the `spark` session already exists and is attached to the cluster,
# so no builder / master / sparkContext is needed (sparkContext is not available on
# standard access mode).

# Data source: Unity Catalog volume
DATA_PATH = "/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed"

# Reading File

**What this cell does:** Defines a DataFrame on the large `transactions.parquet` fact table.

**Core concept:** Reading is **lazy**: only the parquet metadata (schema) is read here. In the plans below this source appears as `Relation [...] parquet` in the logical plans and as `FileScan parquet` in the physical plan.

In [0]:
transactions_file = f"{DATA_PATH}/transactions.parquet"
df_transactions = spark.read.parquet(transactions_file)

**What this cell does:** Shows 5 transaction rows.

**Core concept:** `show()` is an action, so it runs a small job with a limit. Note that every column, including `amt`, is a string; you will see Spark insert `cast(amt as double)` into the plan when we sum it later.

In [0]:
df_transactions.show(5, False)

**What this cell does:** Defines a DataFrame on the small `customers.parquet` dimension table.

**Core concept:** Each read gets its own column ids in the plan (e.g. `cust_id#377` for transactions vs `cust_id#427` for customers). The `#number` is how Catalyst tells apart columns with the same name, which matters when reading join plans.

In [0]:
customers_file = f"{DATA_PATH}/customers.parquet"
df_customers = spark.read.parquet(customers_file)

**What this cell does:** Shows 5 customer rows.

**Core concept:** `age` is a string and `name` holds both first and last name. Both affect the plans below: arithmetic on `age` adds a cast, and filtering on `cast(age as int)` blocks predicate pushdown.

In [0]:
df_customers.show(5, False)

# Spark's Query Plans

Image Credits: `Databricks`

(The original image `../data/spark-execution.png` is not in the workspace; the pipeline it shows is summarised here.)

| Step | Plan in `explain(True)` | What happens |
| --- | --- | --- |
| 1 | `== Parsed Logical Plan ==` | Your DataFrame code turned into a tree of operators; column and table names are not checked yet (unresolved). |
| 2 | `== Analyzed Logical Plan ==` | The analyzer resolves names and types against the catalog/schema (e.g. `city#73` gets an id and a type). |
| 3 | `== Optimized Logical Plan ==` | The Catalyst optimizer applies rules: predicate pushdown, column pruning, constant folding, combining filters and projections. |
| 4 | `== Physical Plan ==` | One or more physical plans are generated and the cheapest is picked (e.g. SortMergeJoin vs BroadcastHashJoin), then whole-stage code generation compiles it to Java bytecode. |

# Narrow Transformations
- `filter` rows where `city='boston'`
- `add` a new column: adding `first_name` and `last_name`
- `alter` an exisitng column: adding 5 to `age` column
- `select` relevant columns

**What this cell does:** Builds the narrow transformation chain, shows 5 rows and prints all four plans with `explain(True)`.

**Core concept: how to read the four plans** (read each one bottom-up, from the source to the output):
* **Parsed**: one `Project` per `withColumn` / `select`, with unresolved names (`'cust_id`).
* **Analyzed**: names resolved to ids and types (`age: bigint`, because string `age` + 5 is cast to bigint).
* **Optimized**: Catalyst **collapses** the stacked `Project`s into one, adds `isnotnull(city)` to the filter and prunes unused columns (`zip` disappears).
* **Physical**: `FileScan parquet` with `PushedFilters: [IsNotNull(city), EqualTo(city,boston)]` (the filter is pushed into the parquet reader), then `ColumnarToRow`, `Filter`, `Project`.

The `*(1)` prefix means these operators are fused into **whole-stage codegen stage 1**. There is no `Exchange`, so this narrow plan runs as a single stage with no shuffle.

In [0]:
df_narrow_transform = (
    df_customers
    .filter(F.col("city") == "boston")
    .withColumn("first_name", F.split("name", " ").getItem(0))
    .withColumn("last_name", F.split("name", " ").getItem(1))
    .withColumn("age", F.col("age") + F.lit(5))
    .select("cust_id", "first_name", "last_name", "age", "gender", "birthday")
)

df_narrow_transform.show(5, False)
df_narrow_transform.explain(True)

# Wide Transformations
1. Repartition
2. Coalesce
3. Joins
4. GroupBy
   - `count`
   - `countDistinct`
   - `sum`

## 1. Repartition

In [0]:

print(spark.conf.get('spark.sql.files.maxPartitionBytes'))
print(int(spark.conf.get('spark.sql.files.maxPartitionBytes').rstrip('bB')))
# The config value is in bytes (e.g. 134217728).
# To convert bytes → megabytes we divide by 1024 twice:
#   ÷ 1024  →  kilobytes (KB)
#   ÷ 1024  →  megabytes (MB)
# This is equivalent to dividing by 1,048,576 (1024²).
print(int(spark.conf.get('spark.sql.files.maxPartitionBytes').rstrip('bB')) / (1024 * 1024), 'MB')

**What this cell does:** Counts how many partitions the transactions DataFrame has when read.

**Core concept:** One partition = one task. For files, the count depends on total size and `spark.sql.files.maxPartitionBytes` (128 MB by default here; the DAGs notebook sets 256 MB, so it gets fewer partitions). `df.rdd.getNumPartitions()` is not available on Spark Connect, so this counts distinct `spark_partition_id()` values, which counts only non-empty partitions.

In [0]:
# df.rdd is not available on standard access mode (Spark Connect); count non-empty partitions instead
num_partitions = df_transactions.select(F.spark_partition_id().alias("pid")).distinct().count()

max_partition_bytes = int(spark.conf.get('spark.sql.files.maxPartitionBytes').rstrip('bB'))
max_partition_mb = max_partition_bytes / (1024 * 1024)

print(f"Number of partitions: {num_partitions}")
print(f"Max partition size:    {max_partition_mb} MB")
print(f"Estimated total size:  ≤ {num_partitions} × {max_partition_mb} MB = {num_partitions * max_partition_mb} MB")
print()
print("⚠️  Note: 12 partitions does NOT necessarily mean 12 × 128 MB.")
print("   spark.sql.files.maxPartitionBytes (128 MB) is the UPPER LIMIT per partition.")
print("   The last partition (and others) may be smaller, so the actual file size")
print("   could be anywhere from just over 11 × 128 MB up to 12 × 128 MB.")
print(f"   Actual file size: between ~{(num_partitions - 1) * max_partition_mb} MB and {num_partitions * max_partition_mb} MB.")

**What this cell does:** Lists every parquet file under `transactions.parquet` and adds up their sizes to get the real on-disk size, then compares it with the partition count.

**Core concept:** The partition count alone cannot tell you the data size. When packing files into partitions, Spark adds `spark.sql.files.openCostInBytes` (4 MB) per file to account for the cost of opening it. With many small files, that padding inflates the bytes per partition, so `num_partitions × maxPartitionBytes` overestimates the real size. Parquet is also compressed, so the size in memory is larger than the on-disk size.

In [0]:
def parquet_size(path):
    """Recursively sum the size (bytes) and count of .parquet files under a path."""
    total_bytes, n_files = 0, 0
    for f in dbutils.fs.ls(path):
        if f.isDir():
            b, n = parquet_size(f.path)
            total_bytes += b
            n_files += n
        elif f.name.endswith(".parquet"):
            total_bytes += f.size
            n_files += 1
    return total_bytes, n_files

total_bytes, n_files = parquet_size(f"{DATA_PATH}/transactions.parquet")

max_partition_bytes = int(spark.conf.get("spark.sql.files.maxPartitionBytes").rstrip("bB"))
open_cost_bytes = 4 * 1024 * 1024  # default spark.sql.files.openCostInBytes
padded_bytes = total_bytes + n_files * open_cost_bytes

MB = 1024 * 1024
print(f"Parquet files:                 {n_files}")
print(f"Actual size on disk:           {total_bytes:,} bytes = {total_bytes / MB:.1f} MB")
print(f"Average file size:             {total_bytes / n_files / MB:.1f} MB")
print(f"Size incl. 4 MB open cost/file: {padded_bytes / MB:.1f} MB")
print(f"Expected partitions:           ~{padded_bytes / max_partition_bytes:.1f} (padded size / {max_partition_bytes / MB:.0f} MB)")
print(f"Observed partitions:           {num_partitions}")

**What these cells do:** Save the transactions data as a managed Delta table, then compare file count, size and read partitions with the raw parquet folder, before and after `OPTIMIZE`.

**Core concept:** Delta stores its data as parquet files, so Spark groups them into partitions the same way (file size + 4 MB open cost, packed up to `maxPartitionBytes`). What changes is the **number and size of files**: Databricks may write larger files (optimized writes), and `OPTIMIZE` compacts small files. Fewer, larger files mean less open-cost padding, so the partition count gets close to `size / maxPartitionBytes`. `DESCRIBE DETAIL` reports `numFiles` and `sizeInBytes` for the **current** table version only.

In [0]:
DELTA_TABLE = "lingaro_sandbox_ne.sandani_sandbox.transactions_delta"

(
    df_transactions
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(DELTA_TABLE)
)

In [0]:
MB = 1024 * 1024
max_partition_bytes = int(spark.conf.get("spark.sql.files.maxPartitionBytes").rstrip("bB"))
open_cost_bytes = 4 * MB  # default spark.sql.files.openCostInBytes

def delta_stats(table):
    """Files, size and read partitions for the current version of a Delta table."""
    d = spark.sql(f"DESCRIBE DETAIL {table}").select("numFiles", "sizeInBytes").first()
    parts = spark.table(table).select(F.spark_partition_id()).distinct().count()
    return d["numFiles"], d["sizeInBytes"], parts

def print_row(label, n_files, size_bytes, parts):
    expected = (size_bytes + n_files * open_cost_bytes) / max_partition_bytes
    print(f"{label:<22}{n_files:>7}{size_bytes / MB:>12.1f}{size_bytes / n_files / MB:>12.1f}{expected:>12.1f}{parts:>10}")

print(f"{'Source':<22}{'Files':>7}{'Size MB':>12}{'Avg MB':>12}{'Expected':>12}{'Actual':>10}")
print_row("Parquet folder", n_files, total_bytes, num_partitions)
print_row("Delta (as written)", *delta_stats(DELTA_TABLE))

**What these cells do:** Compact the Delta table with `OPTIMIZE` and compare again.

**Core concept:** `OPTIMIZE` rewrites small files into larger ones and commits a new table version. The old files stay in storage until `VACUUM` removes them, but Delta reads only the files in the current version, which is why `DESCRIBE DETAIL` rather than a folder listing is the right way to measure a Delta table.

In [0]:
display(spark.sql(f"OPTIMIZE {DELTA_TABLE}"))

print(f"{'Source':<22}{'Files':>7}{'Size MB':>12}{'Avg MB':>12}{'Expected':>12}{'Actual':>10}")
print_row("Parquet folder", n_files, total_bytes, num_partitions)
print_row("Delta (optimized)", *delta_stats(DELTA_TABLE))

**What this cell does:** Reads the metrics Delta recorded for each commit on `transactions_delta` from the table history. Here there is only version 0 (the initial write): the `OPTIMIZE` found nothing to compact, so it did not create a new version.

**Core concept:** Every Delta commit stores `operationMetrics`: files and bytes written, rows written and files removed. They match what the Spark UI shows for the write job's output, and they persist with the table, so you can check them later without the Spark UI. Task-level details such as input size per task, peak memory, shuffle and spill are only in the Spark UI (Compute > your cluster > Spark UI > SQL / DataFrame or Stages).

In [0]:
history = spark.sql(f"DESCRIBE HISTORY {DELTA_TABLE}")

display(
    history.select(
        "version",
        "timestamp",
        "operation",
        F.col("operationMetrics")["numFiles"].alias("files_written"),
        F.col("operationMetrics")["numOutputRows"].alias("output_rows"),
        F.round(F.col("operationMetrics")["numOutputBytes"].cast("long") / (1024 * 1024), 1).alias("output_mb"),
        "operationMetrics",
    )
    .orderBy("version")
)

**What this cell does:** Prints the plans for `repartition(24)`.

**Core concept:** `repartition(n)` is a **wide** transformation: it does a full shuffle that spreads rows evenly over `n` partitions. In the logical plan it is `Repartition 24, true` (`true` = shuffle). In the physical plan it becomes `Exchange RoundRobinPartitioning(24), REPARTITION_BY_NUM`: rows are dealt round-robin across partitions, with no key involved.

`AdaptiveSparkPlan isFinalPlan=false` means **Adaptive Query Execution (AQE)** is on: what you see is the *initial* plan, and AQE may change it at runtime (for example coalescing small shuffle partitions) once real statistics are known. The final plan is visible in the Spark UI after the query runs.

In [0]:
df_transactions.repartition(24).explain(True)

## 2. Coalesce

**What this cell does:** Prints the plans for `coalesce(5)`.

**Core concept:** `coalesce(n)` only **reduces** the number of partitions by merging existing ones on the same executors, without a shuffle. The logical plan shows `Repartition 5, false` (`false` = no shuffle), and the physical plan has a `Coalesce 5` node with **no `Exchange`** and no `AdaptiveSparkPlan` wrapper, because there is no shuffle for AQE to adapt. It is cheaper than `repartition`, but it can leave partitions uneven, and it reduces parallelism for everything upstream in the same stage.

In [0]:
df_transactions.coalesce(5).explain(True)

### Why doesn't `.coalesce()` explicitly show the partitioning scheme?

`.coalesce` doesn't show the partitioning scheme e.g. `RoundRobinPartitioning` because: 
- The operation only minimizes data movement by merging into fewer partitions, it doesn't do any shuffling.
- Because no shuffling is done, the partitioning scheme remains the same as the original DataFrame and Spark doesn't include it explicitly in it's plan as the partitioning scheme is unaffected by `.coalesce`

## 3. Joins

**What this cell does:** Disables automatic broadcast joins (`-1`), so the planner must use a sort-merge join.

**Core concept:** By default Spark broadcasts a join side smaller than 10 MB. Disabling it lets us see the general-purpose **sort-merge join** plan, which is used when both sides are large.

In [0]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

**What this cell does:** Defines an inner join of transactions and customers on `cust_id` (lazy).

**Core concept:** `on="cust_id"` produces a `UsingJoin` in the parsed plan, which keeps a single `cust_id` column. Both tables have `city`, so the result has two `city` columns (`city#387` and `city#433`).

In [0]:
df_joined = (
    df_transactions.join(
        df_customers,
        how="inner",
        on="cust_id"
    )
)

**What this cell does:** Prints the plans for the join.

**Core concept: reading a sort-merge join plan.**
* **Optimized logical plan**: Catalyst adds `Filter isnotnull(cust_id)` on both sides, because null keys can never match in an inner join. It is pushed into the scans as `PushedFilters: [IsNotNull(cust_id)]`.
* **Physical plan**, bottom-up for each branch (`:-` = left, `+-` = right): `FileScan`, then `Filter`, then `Exchange hashpartitioning(cust_id, 200)`, then `Sort [cust_id ASC]`, and both branches meet in `SortMergeJoin`.

`hashpartitioning(cust_id, 200)` means both tables are shuffled into 200 partitions (`spark.sql.shuffle.partitions`) by hash of the key, so equal keys land in the same partition. Each `Exchange` is a stage boundary. `ENSURE_REQUIREMENTS` means the shuffle was added to satisfy the join's required distribution.

In [0]:
df_joined.explain(True)

## 4. GroupBy

**What this cell does:** Prints the transactions schema.

**Core concept:** `printSchema()` needs no job; it only asks the driver to analyze the plan. On Spark Connect it is also a quick way to catch column-name errors early, since analysis is otherwise deferred until an action.

In [0]:
df_transactions.printSchema()

### GroupBy Count

**What this cell does:** Defines a count of transactions per `city` (lazy).

**Core concept:** `groupBy` is wide, since all rows of a city must end up together, but Spark avoids shuffling raw rows by aggregating in two phases. The next cell shows how.

In [0]:
df_city_counts = (
    df_transactions
    .groupBy("city")
    .count()
)

**What this cell does:** Prints the plans for the city count.

**Core concept: two-phase aggregation**, read bottom-up:
1. `FileScan parquet [city]`: column pruning, so only `city` is read (`ReadSchema: struct<city:string>`).
2. `HashAggregate(... partial_count(1))`: each task counts its own rows per city.
3. `Exchange hashpartitioning(city, 200)`: only the small partial counts are shuffled.
4. `HashAggregate(... finalmerge_count(...))`: partial counts are merged into the final count per city.

In [0]:
df_city_counts.explain(True)

**What this cell does:** Defines the total `amt` per city (lazy).

**Core concept:** `agg()` lets you name the output (`txn_amt`) and combine several aggregations in one pass. Because `amt` is a string, the analyzer will insert an implicit cast.

In [0]:
df_txn_amt_city = (
    df_transactions
    .groupBy("city")
    .agg(F.sum("amt").alias("txn_amt"))
)

**What this cell does:** Prints the plans for the city sum.

**Core concept:** This is the same two-phase shape as `count`, with `partial_sum` before the `Exchange` and `finalmerge_sum` after it. The analyzed plan shows `sum(cast(amt as double))`: Spark silently converted the string column. Implicit casts like this are worth spotting in plans; in real code, cast explicitly (ideally to `decimal` for money). Only `[amt, city]` are read from parquet.

In [0]:
df_txn_amt_city.explain(True)

### GroupBy Count Distinct 

**What this cell does:** For each customer, counts how many distinct cities they transacted in (lazy). The variable is named `df_txn_per_city`, but the result is cities per customer.

**Core concept:** A distinct count cannot simply add up partial counts, because the same city can appear in many partitions for one customer. Spark first has to de-duplicate `(cust_id, city)` pairs.

In [0]:
df_txn_per_city = (
    df_transactions
    .groupBy("cust_id")
    .agg(F.countDistinct("city").alias("city_count"))
)

**What this cell does:** Shows 5 results and prints the plans for the distinct count.

**Core concept:** The physical plan has **four HashAggregates and two Exchanges**, read bottom-up:
1. `HashAggregate(keys=[cust_id, city])`: local de-duplication of pairs.
2. `Exchange hashpartitioning(cust_id, city, 200)`: shuffle so each pair lives in one partition.
3. `HashAggregate(keys=[cust_id, city])`: global de-duplication.
4. `HashAggregate(... partial_count(distinct city))`, then `Exchange hashpartitioning(cust_id, 200)`, then `finalmerge_count`: count distinct cities per customer.

Two shuffles instead of one is why `countDistinct` costs more than `count`. `F.approx_count_distinct` is a cheaper alternative when an estimate is enough.

In [0]:
df_txn_per_city.show(5, False)
df_txn_per_city.explain(True)

# 5. Interesting Observations

### Why is a filter step present despite predicate pushdown? 

This is largely due to the way `Spark's Catalyst Optimizer` works. Specifically, due to two separate stages of the query optimization process: Physical Planning and Logical Planning.

- **Logical Planning**: Catalyst optimizer simplifies the unresolved logical plan (which represents the user's query) by applying various rule-based optimizations. This includes `predicate pushdown`, `projection pushdown` where filter conditions and column projections are moved as close to the data source as possible.

- **Physical Planning** phase is where the logical plan is translated into one or more physical plans, which can actually be executed on the cluster. This includes operations like file `scans`, `filters`, `projections`, etc.

In this case, during the logical planning phase, the predicate (`F.col("city") == "boston"`) has been pushed down and will be applied during the scan of the Parquet file (`PushedFilters: [IsNotNull(city), EqualTo(city,boston)]`), thus improving performance.

Now, during the physical planning phase, the same filter condition (`+- *(1) Filter (isnotnull(city#73) AND (city#73 = boston))`) is applied again to the data that's been loaded into memory. This is because of the following reasons:

1. **Guaranteed Correctness:** It might seem **redundant**, but remember that not all data sources can handle pushed-down predicates, and not all predicates can be pushed down. Therefore, **even if a predicate is pushed down to the data source, Spark still includes the predicate in the physical plan** to cover cases where the data source might not have been able to fully apply the predicate. This is Spark's way of making sure the correct data is always returned, no matter the capabilities of the data source.

2. **No Assumptions**: Spark's Catalyst optimizer doesn't make assumptions about the data source's ability to handle pushed-down predicates. The optimizer aims to generate plans that return correct results across a wide range of scenarios. Even if the filter is pushed down, Spark does not have the feedback from data source whether the pushdown was successful or not, so it includes the filter operation in the physical plan as well.

It is more of a **fail-safe mechanism** to ensure data **integrity** and **correctness**.

---

### In what cases will predicate pushdown not work?

2 Examples where **filter pushdown** will not work:

1. **Complex Data Types**: Spark's Parquet data source does not push down filters that involve **complex types**, such as **arrays**, **maps**, and **structs**. This is because these complex data types can have complicated nested structures that the Parquet reader cannot easily filter on.

Here's an example:

```
root
 |-- Name: string (nullable = true)
 |-- properties: map (nullable = true)
 |    |-- key: string
 |    |-- value: string (valueContainsNull = true)

+----------+-----------------------------+
|Name      |properties                   |
+----------+-----------------------------+
|Afaque    |[eye -> black, hair -> black]|
|Naved     |[eye ->, hair -> brown]      |
|Ali       |[eye -> black, hair -> red]  |
|Amaan     |[eye -> grey, hair -> grey]  |
|Omaira    |[eye -> , hair -> brown]     |
+----------+-----------------------------+
```

```python
df.filter(df.properties.getItem("eye") == "brown").show()
```

```
== Physical Plan ==
*(1) Filter (metadata#123[key] = value)
+- *(1) ColumnarToRow
   +- FileScan parquet [id#122,metadata#123] Batched: true, DataFilters: [(metadata#123[key] = value)], Format: Parquet, ...
```

------------------------------------------------

3. Unsupported Expressions: 

In Spark, `Parquet` data source does not support pushdown for filters involving a `.cast` operation. The reason for this behaviour is as follows:
- `.cast` changes the datatype of the column, and the Parquet data source may not be able to perform the filter operation correctly on the cast data.

**Note**: This behavior may vary based on the data source. For example, if you're working with a JDBC data source connected to a database that supports SQL-like operations, the `.cast` filter could potentially be pushed down to the database.

### Example of operation where filter pushdown doesn't work

**What this cell does:** Filters customers with `cast(age as int) > 50`, shows 5 rows and prints the plans.

**Core concept:** Compare the `FileScan` line with the Boston filter earlier:
* `DataFilters: [isnotnull(age), (cast(age as int) > 50)]` lists all filters on data columns.
* `PushedFilters: [IsNotNull(age)]` lists **only the null check**; the `cast(...) > 50` predicate is **not** pushed to the parquet reader.

Parquet can only evaluate simple comparisons on the stored column. Because `age` is stored as a string, the cast has to happen in Spark after the rows are read, so row-group skipping can't help. The fix is to store `age` as an integer so that `age > 50` can be pushed down.

In [0]:
df_customer_gt_50 = (
    df_customers
    .filter(F.col("age").cast("int") > 50)
)
df_customer_gt_50.show(5, False)
df_customer_gt_50.explain(True)